# GWAS tutorial

This notebook is a **tutorialized** version of the BRIDGE `variant_aware.py` entry point.

- The **code blocks below are copied verbatim** from `variant_aware.py` (i.e., code is consistent with the repository script).
- The notebook adds **step-by-step Markdown explanations**, plus **runnable demos** for parsing/mutation logic.
- End-to-end scoring requires a configured BRIDGE environment (dependencies, transformer path, and `.pth` checkpoints).

## CLI arguments

### Common arguments
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--variation_mode` | `before|after` | yes | Score reference window ('before') or ALT-substituted window ('after'). |
| `--fasta_sequence_path` | `PATH` | yes | Input FASTA containing window sequences. |
| `--variant_out_file` | `PATH` | yes | Output file (appended). |
| `--Transformer_path` | `PATH` | yes | Transformer path used by build_Transformer_embeddings. |
| `--model_save_path` | `PATH` | yes | Directory with BRIDGE checkpoints (*.pth). |
| `--device` | `cuda|cuda:N|cpu` | no | Torch device (default: cuda if available else cpu). |

### Pipeline selection flags
| Argument | Type / choices | Required | Meaning |
|---|---:|:---:|---|
| `--gwas` | `-` | no | Force GWAS pipeline (default if no pipeline flag provided). |
| `--ribosnitch/--ribosnitches` | `-` | no | Enable ribosnitch pipeline. |
| `--catalog_variants/--genomic_variants` | `-` | no | Enable catalog-variants pipeline (ClinVar/TCGA/1000G). |

## Input FASTA requirements

- **Sequence**: fixed-length window (commonly ~101 nt).
- **Header** must encode `(variant_pos, ref, alt, strand, seq_start)` for `parse_variant_block`.
  The 0-based in-window index is `idx0 = variant_pos - seq_start`.


## Output format

```
<header_without_>\tPrediction_score:<float>
```


## Script code

### Imports

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

from __future__ import annotations

import argparse
import logging
import os
import re
import sys
from pathlib import Path
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel

repo_root = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(repo_root))

from utils.BRIDGE import BRIDGE
from utils.gen_transformer_embedding import build_Transformer_embeddings
from utils.train_loop import validate_without_sigmoid
from utils.utils import RBPInferDataset
from utils.FeatureEncoding import dealwithdata2
from utils.variant import read_fasta, open_output, parse_variant_block, apply_complement, substitute_base, ModelHub, parse_variant_block_flexible

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### GWAS pipeline function

In [2]:
def process_sequences_gwas(
    names: List[str],
    seqs: List[str],
    args: argparse.Namespace,
    hub: ModelHub,
) -> None:
    """Process each FASTA record and append GWAS/BRIDGE variant-aware predictions.

    For each (header, seq):
      1) Parse (variant_pos, REF, ALT, strand, seq_start) from header.
      2) If strand is '-', complement REF/ALT bases.
      3) Compute 0-based index in the window: idx0 = variant_pos - seq_start.
      4) Validate idx0 bounds and REF base match inside the window.
      5) Choose sequence:
         - before: `modified_seq = seq`
         - after : `modified_seq = seq` with idx0 substituted to ALT
      6) Build multimodal inputs expected by BRIDGE:
         - transformer embedding, attn, struct, motif priors, and biochemical features.
      7) Load checkpoint named by FASTA stem, run `validate_without_sigmoid`, write output.

    Important implied constraints
    -----------------------------
    - The fixed tensors `(1, 1, 101)` imply window length ≈ 101 in typical usage.
    """
    out_fp = open_output(Path(args.variant_out_file))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2.0, device=hub.device))

    with out_fp.open("a") as fout:
        for header, seq in zip(names, seqs):
            try:
                var_pos, ref, alt, strand, seq_start = parse_variant_block(header)
            except ValueError as err:
                logging.error("%s → %s", header, err)
                continue

            if strand == "-":
                ref, alt = apply_complement(ref), apply_complement(alt)

            idx0 = var_pos - seq_start
            if idx0 < 0 or idx0 >= len(seq):
                logging.error("Variant index out of bounds (%s)", header)
                continue
            if seq[idx0] != ref:
                logging.error("Ref base mismatch (%s) — skip", header)
                continue

            modified_seq = substitute_base(seq, idx0, alt) if args.variation_mode == "after" else seq

            test_emb, _ = build_Transformer_embeddings(
                sequences=[modified_seq],
                transformer_path=str(args.Transformer_path),
                device=hub.device,
                k=1,
                transpose_to_ch_first=True,
            )
            N = int(test_emb.shape[0])
            test_attn = np.zeros((N, 101, 103))
            struct = np.zeros((N, 1, 101))
            motif = np.zeros((N, 1, 101))
            bio_chem = dealwithdata2(modified_seq).transpose([0, 2, 1])

            dataset = RBPInferDataset(
                embedding=test_emb,
                attn=test_attn,
                struct=struct,
                motif=motif,
                biochem=bio_chem,
            )
            loader = DataLoader(dataset, batch_size=1, shuffle=False)

            filename_stem = Path(args.fasta_sequence_path).stem
            bridge = hub.load_bridge(Path(args.model_save_path), filename_stem)
            if bridge is None:
                continue

            prob = validate_without_sigmoid(bridge, hub.device, loader, criterion).item()
            fout.write(f"{header.lstrip('>')}\tPrediction_score:{prob:.6f}\n")

### CLI + main

In [3]:
# Backward-compatible alias: keep the old function name if other scripts import it.
process_sequences = process_sequences_gwas

def build_argparser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Variant-aware scoring with BRIDGE. Supports three pipelines:\n"
            "  (1) GWAS windows (legacy variant_aware.py behavior; default)\n"
            "  (2) Ribosnitch scoring (BRIDGE; per-record checkpoint selection)\n"
            "  (3) Catalog variants (ClinVar/TCGA/1000G-style FASTA batches; SNVs)\n"
        )
    )

    # ------------------------------------------------------------------
    # Common arguments (shared across pipelines)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--variation_mode",
        choices=["before", "after"],
        required=True,
        help="Score sequences before variation (reference) or after variation (mutated).",
    )
    parser.add_argument("--fasta_sequence_path", required=True, type=Path)
    parser.add_argument("--variant_out_file", required=True, type=Path)
    parser.add_argument("--Transformer_path", required=True, type=Path)
    parser.add_argument("--model_save_path", required=True, type=Path)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------------------------------------------------
    # Pipeline selection flags
    #
    # Backward compatibility:
    # - If you pass none of these flags, we default to GWAS mode.
    # - `--ribosnitches` is accepted as an alias for `--ribosnitch`.
    # - `--genomic_variants` is accepted as an alias for `--catalog_variants`.
    # ------------------------------------------------------------------
    pipe = parser.add_mutually_exclusive_group(required=False)
    pipe.add_argument(
        "--gwas",
        action="store_true",
        help="Force GWAS window scoring (default if no pipeline flag is provided).",
    )
    pipe.add_argument(
        "--ribosnitch",
        "--ribosnitches",
        dest="ribosnitch",
        action="store_true",
        help="Run ribosnitch scoring (BRIDGE).",
    )
    pipe.add_argument(
        "--catalog_variants",
        "--genomic_variants",
        dest="catalog_variants",
        action="store_true",
        help="Run ClinVar/TCGA/1000G-style FASTA batch scoring (SNVs).",
    )

    # ------------------------------------------------------------------
    # Ribosnitch-specific options
    # ------------------------------------------------------------------
    parser.add_argument(
        "--ribosnitch_after_variation",
        "--ribosnitches_after_variation",
        dest="ribosnitch_after_variation",
        action="store_true",
        help="Force ALT substitution for ribosnitch scoring, regardless of --variation_mode.",
    )
    parser.add_argument(
        "--ribosnitch_out_dir",
        "--ribosnitches_out_dir",
        dest="ribosnitch_out_dir",
        type=Path,
        default=Path("./results/ribosnitches"),
        help="Root output directory for ribosnitch results.",
    )

    # ------------------------------------------------------------------
    # Catalog-variants options (also used by genomic_variants.py)
    # ------------------------------------------------------------------
    parser.add_argument(
        "--model_id_strategy",
        choices=["from_header", "from_fasta_stem"],
        default="from_header",
        help=(
            "How to choose checkpoint name for catalog variants. "
            "from_header: <PROTEIN>_<CELL> parsed from header; "
            "from_fasta_stem: use FASTA filename stem."
        ),
    )
    parser.add_argument(
        "--k",
        type=int,
        default=1,
        help="K-mer / stride parameter forwarded to build_Transformer_embeddings (catalog variants branch).",
    )
    parser.add_argument(
        "--pos_weight",
        type=float,
        default=2.0,
        help="Positive class weight for BCEWithLogitsLoss (catalog variants branch).",
    )
    parser.add_argument(
        "--strict_ref_match",
        action="store_true",
        help="If set, skip records when REF/ALT cannot be matched inside the window (catalog variants branch).",
    )
    parser.add_argument(
        "--disable_off_by_one",
        action="store_true",
        help="Disable +/-1 position fallback when locating the SNV inside the window (catalog variants branch).",
    )

    return parser

def main() -> None:
    args = build_argparser().parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    device = torch.device(args.device)

    logging.info("Loading FASTA from %s", args.fasta_sequence_path)
    headers, sequences = read_fasta(args.fasta_sequence_path)

    # Pipeline selection validation.
    # - Default: GWAS (for backward compatibility) when no pipeline flag is provided.
    selected = []
    if bool(getattr(args, "gwas", False)):
        selected.append("gwas")
    if bool(getattr(args, "ribosnitch", False)) or bool(getattr(args, "ribosnitch_after_variation", False)):
        selected.append("ribosnitch")
    if bool(getattr(args, "catalog_variants", False)):
        selected.append("catalog_variants")

    if len(selected) > 1:
        raise SystemExit(f"Conflicting pipeline flags: {selected}. Please choose only one.")

    pipeline = selected[0] if selected else "gwas"

    hub = ModelHub(args.Transformer_path, device)

    if pipeline == "gwas":
        logging.info("Running GWAS pipeline (%s_variation)", args.variation_mode)
        process_sequences_gwas(headers, sequences, args, hub)

    logging.info("Finished. Results appended to %s", args.variant_out_file)


In [ ]:
from pathlib import Path
import os
import logging
import torch

def find_bridge_root(start: Path) -> Path:
    # Search upwards for typical BRIDGE repo markers
    for p in [start, *start.parents]:
        if (p / "variant_aware.py").exists() and (p / "utils" / "variant.py").exists():
            return p
    raise FileNotFoundError(
        f"Cannot locate BRIDGE repo root from: {start}\n"
        f"Expected to find: variant_aware.py and utils/variant.py in the repo root."
    )

# Locate repo root robustly (works even if notebook cwd is not the repo root)
BRIDGE_HOME = find_bridge_root(Path.cwd())
os.chdir(BRIDGE_HOME)

# Keep paths consistent with your bash script
TRANSFORMER_PATH = str((BRIDGE_HOME / "RBPformer").resolve())
MODEL_SAVE_PATH = str((BRIDGE_HOME / "results" / "model").resolve())  # if you want the repo model path

# Match the bash inputs
mode = "after"
fasta = str((BRIDGE_HOME / "dataset_variant" / "AUH_HepG2.fa").resolve())

# Match the bash output path
out_path = BRIDGE_HOME / "results" / "reproducibility" / "gwas_resources" / "after_mut" / "AUH_HepG2_after_mut.txt"
out_path.parent.mkdir(parents=True, exist_ok=True)  # mkdir -p for the output directory

# Build args like the CLI call in bash (no --catalog_variants here)
args = build_argparser().parse_args([
    "--variation_mode", mode,
    "--fasta_sequence_path", fasta,
    "--Transformer_path", TRANSFORMER_PATH,
    "--model_save_path", MODEL_SAVE_PATH,
    "--variant_out_file", str(out_path.resolve()),
    "--device", "cuda:3",
])

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
device = torch.device(args.device)

headers, sequences = read_fasta(args.fasta_sequence_path)
hub = ModelHub(args.Transformer_path, device)

process_sequences_gwas(headers, sequences, args, hub)


/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at /fs1/private/user/wangyubo/code/BRIDGE/RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-